# Эксперименты Филиппа на Amazon

Подключите версию данных с каталогом, сплитами и манифестом. Для Kaggle добавьте исходный снимок проекта как вход для ноутбука и задайте пути ниже. Локально можно указать корень репозитория и локальный split. Первый запуск по умолчанию оценивается на validation.

Этот ноутбук запускает одну партию на validation или test; для выбора гиперпараметров и повторов по трём seed используйте scripts/run_study.py по инструкции README.

В таблице результатов HitRate@10 — доля запросов с хотя бы одним релевантным товаром в топ-10.

По умолчанию запускаются только модели Филиппа и дополнительные бейзлайны из configs/filipp_owned.json.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json
import os
import sys

PROJECT_ROOT = Path(os.environ.get('RECSYS_PROJECT_ROOT', '/kaggle/input/complementary-recsys-code'))
SPLIT_DIR = Path(os.environ.get('RECSYS_SPLIT_DIR', '/kaggle/input/amazon-complementary-split'))
OUTPUT_ROOT = Path(os.environ.get('RECSYS_OUTPUT_ROOT', '/kaggle/working/runs'))
CODE_REVISION = os.environ.get('RECSYS_CODE_REVISION')
CONFIG = os.environ.get('RECSYS_CONFIG', 'configs/filipp_owned.json')
PARTITION = os.environ.get('RECSYS_PARTITION', 'validation')

assert (PROJECT_ROOT / 'src/complementary_recsys').is_dir(), PROJECT_ROOT
assert (SPLIT_DIR / 'manifest.json').is_file(), SPLIT_DIR
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

In [ ]:
import torch
from complementary_recsys.data.bundle import SplitBundle
from complementary_recsys.experiment import ExperimentRunner

config = json.loads((PROJECT_ROOT / CONFIG).read_text())
config['training']['device'] = 'cuda' if torch.cuda.is_available() else 'cpu'
if CODE_REVISION:
    config['code_revision'] = CODE_REVISION
run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
output = OUTPUT_ROOT / run_id
bundle = SplitBundle.load(SPLIT_DIR)
results = ExperimentRunner(bundle, config, PARTITION).run(output)
display(results)

In [ ]:
import pandas as pd

recommendations = pd.read_csv(output / 'recommendations.csv.gz', dtype={'source_id': str})
display(recommendations.head(10))
print(output)